# Step 02b — Predictor selection by AICc

Rank **all subsets** of the candidate CTD predictors by **AICc** (small-sample-corrected AIC) and choose the best per response. Run on the full data; the chosen subset is then validated out-of-group in Step 03/04.

AICc answers *which predictors*; leave-one-cast-out answers *does the chosen model predict*. Report both.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import pandas as pd


In [ ]:
# --- parameters ---
MODEL_READY = '../outputs/model_ready.csv'
CRITERION = 'aicc'   # 'aicc' (recommended) | 'aic' | 'bic'


In [ ]:
from ctd_carb_model import data, selection
ds = data.prepare(pd.read_csv(MODEL_READY))
measured = [r for r in ds.responses if ds.response_status[r]=='measured']
chosen = {}
for resp in measured:
    best, ranking = selection.select_predictors(ds.frame, resp, ds.predictors, CRITERION)
    chosen[resp] = best
    print(f'=== {resp} ===')
    print(ranking.head(5)[['predictors','n_predictors','r2','adj_r2',CRITERION,f'delta_{CRITERION}',f'weight_{CRITERION}']].round(3).to_string(index=False))
    print('  ->', selection.interpret(ranking, CRITERION)); print()
import json; json.dump(chosen, open('../outputs/selected_predictors.json','w'), indent=2)
print('selected:', chosen)

## Reading it
- lowest AICc = preferred; **Δ<2** = equivalent support; **Δ>10** = none.
- **weight** = share of support among compared subsets.
- If several subsets are within Δ<2, the data don't decisively prefer one — say so, and consider reporting the candidate set.

To use the selected predictors in the comparison, call `compare.compare_models(..., select_by='aicc')` (Step 04 does this when `SELECT_BY` is set), which selects per response then validates out-of-group.